In [ ]:
from MobilityHubDataObjects import *
from MobilityHubDataObjects.BaseLayer import *
import geopandas as gpd
import folium
import datetime as dt

In [ ]:
CONFIG_MAP_AREA_PATH = "map_area_path"
CONFIG_MAP_AREA_PROJECTED_CRS = "map_area_projected_crs"
CONFIG_GTFS_TRANSITLAND_URL = "gtfs_url"
CONFIG_GTFS_TRANSITLAND_KEY_PATH = "gtfs_key_path"
CONFIG_GTFS_CACHE_FOLDER = "gtfs_cache_folder"
CONFIG_GTFS_OVERRIDE_SHEET = "gtfs_override_sheet"
CONFIG_FTA_FACILITY_INVENTORY_PATH = "fta_facility_inventory_path"
CONFIG_TIGER_STATES_PATH = "tiger_states_path"
CONFIG_CITYBIKES_URL = "citybikes_url"
CONFIG_AFDC_API_KEY = "afdc_api_key"
CONFIG_AFDC_URL = "afdc_url"
CONFIG_OSM_CACHE_FOLDER = "osm_cache_folder"
CONFIG_EPA_EJSCREEN_PATH = "epa_ejscreen_path"
CONFIG_SMART_LOCATION_PATH = "smart_location_path"

GEODESIC_CRS = 4326

WAYNE_COUNTY_PATH = "./rawData/waynecounty.geojson"
WAYNE_COUNTY_GTFS_CACHE = "./cache/gtfs_cache/wayne_county"
WAYNE_COUNTY_CRS = 6498

COOK_COUNTY_PATH = "./rawData/CookCounty.geojson"
COOK_COUNTY_GTFS_CACHE = "./cache/gtfs_cache/chicagoland"
COOK_COUNTY_GTFS_OVERRIDE = "./cache/gtfs_cache/chicagoland/override_feeds.csv"
COOK_COUNTY_CRS = 26971

SANTA_CRUZ_COUNTY_PATH = "./rawData/santacruz_county.geojson"
SANTA_CRUZ_COUNTY_GTFS_CACHE = "./cache/gtfs_cache/santacruz"
SANTA_CRUZ_COUNTY_GTFS_OVERRIDE = "./cache/gtfs_cache/santacruz/override_feeds.csv"
SANTA_CRUZ_COUNTY_CRS = 26943

LOS_ANGELES_COUNTY_PATH = "./rawData/losangelescounty.geojson"
LOS_ANGELES_COUNTY_GTFS_CACHE = "./cache/gtfs_cache/losangeles"
LOS_ANGELES_COUNTY_GTFS_OVERRIDE = None
LOS_ANGELES_COUNTY_CRS = 6423

ANTELOPE_VALLEY_PATH = "./rawData/av_test_area.geojson"
ANTELOPE_VALLEY_GTFS_CACHE = "./cache/gtfs_cache/antelope_valley_test"

DC_PATH = "./rawData/dc.geojson"
DC_GTFS_CACHE = "./cache/gtfs_cache/dc"
DC_GTFS_OVERRIDE = None
DC_CRS = 26985

# COMPLETE CONFIG HERE
CONFIG = {
    CONFIG_MAP_AREA_PATH: COOK_COUNTY_PATH,
    CONFIG_MAP_AREA_PROJECTED_CRS: COOK_COUNTY_CRS, # This should be an epsg number with the units in meters
    CONFIG_GTFS_TRANSITLAND_URL: "https://transit.land/api/v2/rest/feeds.json",
    CONFIG_GTFS_TRANSITLAND_KEY_PATH: "./rawData/TRANSITLAND_KEY",
    CONFIG_GTFS_CACHE_FOLDER: COOK_COUNTY_GTFS_CACHE,
    CONFIG_GTFS_OVERRIDE_SHEET: COOK_COUNTY_GTFS_OVERRIDE,
    CONFIG_FTA_FACILITY_INVENTORY_PATH: "./rawData/2022 Facility Inventory.xlsx",
    CONFIG_TIGER_STATES_PATH: "./rawData/tl_2023_us_state/tl_2023_us_state.shp",
    CONFIG_CITYBIKES_URL: "http://api.citybik.es/",
    CONFIG_OSM_CACHE_FOLDER: "./cache/osmnx_cache",
    CONFIG_AFDC_URL: "https://developer.nrel.gov/api/alt-fuel-stations/v1/nearest.geojson",
    CONFIG_AFDC_API_KEY: "./rawData/AFDC_API_KEY",
    CONFIG_EPA_EJSCREEN_PATH: "rawData/EJScreen_2024_BG_with_AS_CNMI_GU_VI.gdb",
    CONFIG_SMART_LOCATION_PATH: "rawData/SmartLocationDatabaseV3/SmartLocationDatabase.gdb",
}

In [ ]:
# Define map area
map_area = gpd.read_file(CONFIG[CONFIG_MAP_AREA_PATH]).to_crs(GEODESIC_CRS).loc[0,"geometry"]
map_area

In [ ]:
# Instantiate data objects

gtfs_instance = GTFSDataObject(
    CONFIG[CONFIG_GTFS_CACHE_FOLDER],
    CONFIG[CONFIG_GTFS_TRANSITLAND_URL],
    dt.timedelta(days=100),
    dt.time(hour=10), #TODO: need to handle tz
    dt.time(hour=15),
    min_headway=9000,
    api_key_path=CONFIG[CONFIG_GTFS_TRANSITLAND_KEY_PATH],
    gtfs_override_feeds_path=CONFIG[CONFIG_GTFS_OVERRIDE_SHEET]
)
fta_instance = FTAFacilityInventoryDataObject(
    CONFIG[CONFIG_FTA_FACILITY_INVENTORY_PATH],
    CONFIG[CONFIG_TIGER_STATES_PATH],
    CONFIG[CONFIG_OSM_CACHE_FOLDER]
)
citybikes_instance = CityBikesDataObject(CONFIG[CONFIG_CITYBIKES_URL])
afdc_instance = AFDCApiDataObject(CONFIG[CONFIG_AFDC_URL], CONFIG[CONFIG_AFDC_API_KEY],CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS])
osm_bike_parking_instance = OSMBikeParkingDataObject(CONFIG[CONFIG_OSM_CACHE_FOLDER], {"amenity": ["bicycle_parking"]})
#TODO: need to find a way to get the gdf in here, not sure how is best to do that
smart_location_info = SmartLocationWrapper(CONFIG[CONFIG_SMART_LOCATION_PATH], CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS])
base_layer_instance = BaseLayer(
    [
        #SmartLocationJobAccessibility(smart_location_info),
        SmartLocationPopulationDensity(smart_location_info),
        SmartLocationJobDensity(smart_location_info),
        SmartLocationRetailEntertainmentJobDensity(smart_location_info),
        SmartLocationRawJobs(smart_location_info),
        CensusModeshare(),
        CensusCarlessness(),
        SmartLocationNationalWalkabilityIndex(smart_location_info),
        BaseLayerEjscreen(CONFIG[CONFIG_EPA_EJSCREEN_PATH]),
    ],
    "rawData/tl_2023_us_county/tl_2023_us_county.shp",
    CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS],
    ColorMaps.NATIONAL_WALKABILITY_INDEX_COLORMAP,
    smooth=True
)
bike_instance = OSMBikeStreetsDataObject(
    CONFIG[CONFIG_OSM_CACHE_FOLDER],
    reference=gtfs_instance,
    local_crs=CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS]
)
mobility_hub_instance = MobilityHubDataObject(
    gtfs_instance,
    base_layer_instance,
    local_crs=CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS]
)

all_objects = {
    "GTFS": gtfs_instance,
    "BASE": base_layer_instance,
    "BIKE": bike_instance,
    "FTA": fta_instance,
    "CITYBIKES": citybikes_instance,
    "AFDC": afdc_instance,
    "OSM BIKE PARKING": osm_bike_parking_instance,
}
objects_load_order = (
    "BASE",
    "BIKE",
    "GTFS",
    "FTA",
    "AFDC",
    "OSM BIKE PARKING",
    "CITYBIKES"
)
objects_must_await = (
    "GTFS"
)
await gtfs_instance.load_data(map_area, GEODESIC_CRS)
base_layer_instance.load_data(map_area, GEODESIC_CRS)

In [ ]:
base_layer_instance.gdf

#### Mobility Hub Map

In [ ]:
# Mobility Hub Only Map
mobility_hub_instance.load_data(map_area, GEODESIC_CRS)
mobility_hub_map = folium.Map(
    location=(map_area.centroid.y, map_area.centroid.x),
    tiles="Cartodb Positron",
    zoom_start=10,
    prefer_canvas=True
)
base_layer_instance.get_folium_plot().add_to(mobility_hub_map)
mobility_hub_instance.get_folium_plot().add_to(mobility_hub_map)
mobility_hub_map

#### All Elements

In [ ]:
# Load data objects
for name, object in all_objects.items():
    print(name)
    if not object.get_is_loaded():
        if name in objects_must_await:
            await object.load_data(map_area, GEODESIC_CRS)
        else:
            object.load_data(map_area, GEODESIC_CRS)

In [ ]:
# Main Map
display_map = folium.Map(
    location=(map_area.centroid.y, map_area.centroid.x),
    tiles="Cartodb Positron",
    zoom_start=10,
    prefer_canvas=True
)
for object in objects_load_order:
    print(object)
    assert all_objects[object].get_is_loaded()
    if len(all_objects[object].gdf) > 0: #TODO: quick fix, add "has_entries" method to each data object instead
        all_objects[object].get_folium_plot().add_to(display_map)
#all_objects["GTFS"].get_folium_plot().add_to(display_map)
display_map

In [ ]:
bike_instance.gdf

In [ ]:
display_map.save("chicago_dec9.html")

In [ ]:
smart_location_info.loaded_county_fips

In [ ]:
from pygris.data import get_lodes, block_groups, tracts

In [ ]:


gdf_2021 = get_lodes(state="IL", year=2021, return_geometry=True)
gdf_2018 = get_lodes(state="IL", year=2018, return_geometry=True)

In [ ]:
cook_county_block_groups_2021 = block_groups(state="IL", county="Cook", year=2021)
cook_county_block_groups_2018 = block_groups(state="IL", county="Cook", year=2018)

In [ ]:
gdf_2021

In [ ]:
jobs_2021 = gdf_2021.groupby("w_geocode")["S000"].sum()
df_2021_processed = gdf_2021.drop_duplicates("w_geocode").set_index("w_geocode")[["geometry"]].to_crs(CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS])
df_2021_processed["jobs_2021"] = jobs_2021
df_2021_processed["original_geoid"] = df_2021_processed.index
df_2021_processed["original_area"] = df_2021_processed.area

jobs_2018 = gdf_2018.groupby("w_geocode")["S000"].sum()
df_2018_processed = gdf_2018.drop_duplicates("w_geocode").set_index("w_geocode")[["geometry"]].to_crs(CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS])
df_2018_processed["jobs_2018"] = jobs_2018

In [ ]:
cc_tracts = tracts(state="IL", county="cook")

In [ ]:
gdf = cc_tracts.sjoin(df_2021_processed.to_crs(cc_tracts.crs).reset_index()[["geometry", "jobs_2021"]], predicate="within").
gdf = gdf.sjoin(df_2018_processed.reset_index()[["geometry", "jobs_2018"]].to_crs(cc_tracts.crs), predicate="within")

In [ ]:
gdf

In [ ]:
(df_2021_processed["jobs_2021"] - df_2018_processed["jobs_2018"]).sort_values()